In [1]:
import os
%pwd

'c:\\Users\\uNUKE\\OneDrive\\Documents\\DSML\\GeoCognition\\prototype'

In [2]:
os.chdir("../")
%pwd

'c:\\Users\\uNUKE\\OneDrive\\Documents\\DSML\\GeoCognition'

In [ ]:
import io
import json
from minio import Minio
from minio.error import S3Error
from dotenv import load_dotenv
from include.common import read_yaml
from include.constants import CONFIG_FILE_PATH
import pandas as pd

load_dotenv()

True

# Reading in Data From MinIO

In [5]:
# Connection settings
endpoint = "localhost:9000"  # Update if using a different host/portq
access_key = os.getenv("MINIO_ACCESS_KEY", "minio")
secret_key = os.getenv("MINIO_SECRET_KEY", "minio123")

client = Minio(
    endpoint,
    access_key=access_key,
    secret_key=secret_key,
    secure=False,
)

BUCKET_NAME = "earthquake-curated"
OBJECT_NAME = "year=2026/month=03/day=18/flattened.parquet"

response = client.get_object(BUCKET_NAME, OBJECT_NAME)
try:
    payload = pd.read_parquet(io.BytesIO(response.read()))
finally:
    response.close()
    response.release_conn()

payload

,type,id,properties.mag,properties.place,properties.time,properties.updated,properties.tz,properties.url,properties.detail,properties.felt,...,properties.dmin,properties.rms,properties.gap,properties.magType,properties.type,properties.title,geometry.type,longitude,latitude,depth_km


In [31]:
from include.constants import EQ_COLUMNS
import numpy as np

df = payload

# Ensure all expected source columns exist and preserve order
for col in EQ_COLUMNS:
    if col not in df.columns:
        df[col] = pd.NA
df = df[EQ_COLUMNS]

# If some rows are JSON strings, parse them first.
def to_list(v):
    if isinstance(v, str):
        try:
            return json.loads(v)   # e.g. "[40.1, 9.2, 10.0]" -> [40.1, 9.2, 10.0]
        except Exception:
            return np.nan
    return v

coords = df["geometry.coordinates"].apply(to_list)

df["longitude"] = coords.str[0]
df["latitude"] = coords.str[1]
df["depth_km"] = coords.str[2]   # optional

df

,type,id,properties.mag,properties.place,properties.time,properties.updated,properties.tz,properties.url,properties.detail,properties.felt,...,properties.rms,properties.gap,properties.magType,properties.type,properties.title,geometry.type,geometry.coordinates,longitude,latitude,depth_km
0,Feature,us6000pi33,4.50,"29 km N of Āwash, Ethiopia",1736121679293,1742247289040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,8.0,...,0.70,116.0,mb,earthquake,"M 4.5 - 29 km N of Āwash, Ethiopia",Point,"[40.1161, 9.2432, 10.0]",40.1161,9.2432,10.000
1,Feature,us6000pkmw,4.40,"237 km WNW of Fangale’ounga, Tonga",1736121783418,1742247289040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,0.70,260.0,mb,earthquake,"M 4.4 - 237 km WNW of Fangale’ounga, Tonga",Point,"[-176.3331, -18.7792, 10.0]",-176.3331,-18.7792,10.000
2,Feature,us6000pi35,4.10,"21 km S of Tokoroa, New Zealand",1736122222062,1742247289040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,1.0,...,1.07,87.0,mb,earthquake,"M 4.1 - 21 km S of Tokoroa, New Zealand",Point,"[175.8386, -38.4268, 218.679]",175.8386,-38.4268,218.679
3,Feature,us6000pkmx,4.00,"6 km SW of Āwash, Ethiopia",1736123851259,1742247289040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,0.80,122.0,mb,earthquake,"M 4.0 - 6 km SW of Āwash, Ethiopia",Point,"[40.1269, 8.9446, 10.0]",40.1269,8.9446,10.000
4,Feature,us6000pi3a,4.30,"29 km SW of Gelemso, Ethiopia",1736124600878,1742247289040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,0.59,107.0,mb,earthquake,"M 4.3 - 29 km SW of Gelemso, Ethiopia",Point,"[40.3056, 8.6588, 10.0]",40.3056,8.6588,10.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,Feature,us6000pkp3,4.30,"28 km SE of Debre Birhan, Ethiopia",1736199979480,1742247299040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,0.73,148.0,mb,earthquake,"M 4.3 - 28 km SE of Debre Birhan, Ethiopia",Point,"[39.7283, 9.5064, 10.0]",39.7283,9.5064,10.000
80,Feature,pr71470298,2.92,"23 km WNW of Rincón, Puerto Rico",1736200284990,1736201883910,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,0.16,301.0,md,earthquake,"M 2.9 - 23 km WNW of Rincón, Puerto Rico",Point,"[-67.4435, 18.437, 4.04]",-67.4435,18.4370,4.040
81,Feature,us6000pi96,2.80,"28 km NE of Ranchester, Wyoming",1736201961973,1742247299040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,0.38,97.0,ml,mining explosion,M 2.8 Mining Explosion - 28 km NE of Rancheste...,Point,"[-106.9096, 45.0868, 0.0]",-106.9096,45.0868,0.000
82,Feature,us7000pale,2.80,"226 km SSE of King Cove, Alaska",1736205127406,1742247299040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,0.32,259.0,ml,earthquake,"M 2.8 - 226 km SSE of King Cove, Alaska",Point,"[-161.1245, 53.1538, 18.736]",-161.1245,53.1538,18.736
